# Module 6 — Notebook 1: Cavity Optomechanics, Radiation Pressure & Dynamical Backaction

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![NumPy](https://img.shields.io/badge/numpy-2.0+-green.svg)](https://numpy.org/)
[![SciPy](https://img.shields.io/badge/scipy-1.12+-teal.svg)](https://scipy.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Publication Quality](https://img.shields.io/badge/Visualizations-Publication%20Quality-indigo.svg)]()

---

### Executive Overview & Abstract

Cavity optomechanics investigates the coupled quantum dynamics of electromagnetic radiation and macroscopic mechanical motion. Mediated by the radiation pressure force of circulating photons:
$$\hat{F}_{\mathrm{rad}} = -\frac{\partial \hat{H}}{\partial \hat{x}} = \hbar G_{\mathrm{om}} \hat{a}^\dagger\hat{a}$$
a moving mechanical boundary (such as a cantilever, membrane, or microtoroid) modulates the optical cavity resonance frequency. In turn, the intracavity photon field exerts a dynamic, retarded radiation pressure force back upon the mechanical oscillator—a phenomenon known as **dynamical backaction**.

Depending on the detuning $\Delta = \omega_d - \omega_0$ of the driving laser relative to the cavity resonance:
1. **Red Sideband Driving ($\Delta \approx -\omega_m$)**: The linearized interaction acts as a state-exchange / beam-splitter Hamiltonian $\hat{H}_{\mathrm{int}} \approx -\hbar G (\hat{d}^\dagger\hat{b} + \hat{d}\hat{b}^\dagger)$. Phonons are coherently up-converted into cavity photons and dissipated into the optical vacuum, enabling **resolved-sideband ground-state cooling** of macroscopic objects ($\bar{n}_f < 1$) and opening an ultra-narrow **Optomechanically Induced Transparency (OMIT)** window.
2. **Blue Sideband Driving ($\Delta \approx +\omega_m$)**: The interaction acts as a parametric amplifier / two-mode squeezer $\hat{H}_{\mathrm{int}} \approx -\hbar G (\hat{d}^\dagger\hat{b}^\dagger + \hat{d}\hat{b})$, amplifying mechanical vibrations, generating quantum entanglement, and driving self-sustained mechanical oscillations (**phonon lasing**).

This notebook develops a comprehensive, publication-grade computational framework for cavity optomechanics. We implement:
- The fundamental non-linear radiation pressure Hamiltonian $\hat{H} = -\hbar g_0 \hat{a}^\dagger\hat{a}(\hat{b} + \hat{b}^\dagger)$ and its multi-photon linearized expansion $G = g_0 \sqrt{\bar{n}_{\mathrm{cav}}}$.
- Exact frequency-dependent susceptibility and Heisenberg-Langevin equations for optical and mechanical fluctuations.
- The **optical spring effect** $\delta\omega_m(\Delta)$ and dynamical backaction damping rate $\Gamma_{\mathrm{opt}}(\Delta)$.
- **Ground-state cooling**: Derivation and numerical verification of the quantum limit $\bar{n}_{\mathrm{min}} = (\kappa / 4\omega_m)^2 \ll 1$ in the resolved-sideband regime ($\omega_m \gg \kappa$).
- **Optomechanically Induced Transparency (OMIT)**: Quantum interference between probe laser and mechanical sidebands.

---

### Pedagogical & Computational Roadmap

1. **Theoretical Foundations**: Radiation pressure coupling, canonical linearization, and dynamical backaction rates.
2. **Computational Engine (`CavityOptomechanicsEngine`)**:
   - Frequency-domain susceptibility models for optical cavity $\chi_c(\omega)$ and mechanical resonator $\chi_m(\omega)$.
   - Retarded radiation pressure self-energy, optical spring shift $\delta\omega_m$, and optomechanical damping $\Gamma_{\mathrm{opt}}$.
   - Exact complex probe reflection and transmission spectra $t_p(\Omega)$ for OMIT.
   - Sideband cooling curves predicting final phonon occupancy $\bar{n}_f$ vs. drive power and bath temperature.
3. **Publication-Grade 4-Panel Visualization**:
   - *Panel (a)*: Optomechanically Induced Transparency (OMIT) probe transmission and steep phase dispersion.
   - *Panel (b)*: Dynamical Backaction: Optomechanical damping $\Gamma_{\mathrm{opt}}$ and Optical Spring $\delta\omega_m$ vs. laser detuning $\Delta$.
   - *Panel (c)*: Sideband Cooling Trajectory: Thermal bath reduction ($n_{\mathrm{th}} = 100 \to \bar{n}_f < 1$) and the quantum backaction limit.
   - *Panel (d)*: Mechanical Displacement Noise Spectrum $S_{xx}(\omega)$ showing quantum radiation pressure backaction noise and linewidth broadening.
4. **Interactive Exploration Suite**: Real-time tuning of laser power, detuning, mechanical $Q$-factor, and bath temperature.
5. **Automated Verification Suite**: Assert exact OMIT peak transmission ($> 95\%$), red/blue damping anti-symmetry, and the fundamental quantum cooling bound $\bar{n}_{\mathrm{min}} = (\kappa / 4\omega_m)^2$.


In [ ]:
# Cell 2: Setup, Imports, and Publication Styling Configuration
import sys
import os
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional, Union

# Set random seed for reproducibility
np.random.seed(42)

# Publication-grade plotting parameters
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.titlesize": 16,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "lines.linewidth": 2.0,
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
})

print(f"NumPy version: {np.__version__}")
print("Cavity Optomechanics environment initialized successfully.")


---

## Theoretical Deep Dive: Cavity Optomechanics & Dynamical Backaction

### 1. The Optomechanical Hamiltonian

Consider a Fabry-Pérot optical cavity of rest length $L$ with one fixed mirror and one movable mechanical mirror of effective mass $m$ and natural resonance frequency $\omega_m$.
The cavity resonance frequency depends parametrically on the mechanical displacement $\hat{x}$:
$$\omega_{\mathrm{cav}}(\hat{x}) \approx \omega_0 - G_{\mathrm{om}} \hat{x}, \quad G_{\mathrm{om}} = -\left.\frac{\partial \omega_{\mathrm{cav}}}{\partial x}\right|_{x=0} = \frac{\omega_0}{L}$$
Quantizing the mechanical harmonic oscillator with zero-point position fluctuations $x_{\mathrm{zpf}} = \sqrt{\frac{\hbar}{2m\omega_m}}$:
$$\hat{x} = x_{\mathrm{zpf}} \left( \hat{b} + \hat{b}^\dagger \right)$$
The bare optomechanical Hamiltonian takes the canonical form:
$$\hat{H} = \hbar \omega_0 \hat{a}^\dagger\hat{a} + \hbar \omega_m \hat{b}^\dagger\hat{b} - \hbar g_0 \hat{a}^\dagger\hat{a} \left( \hat{b} + \hat{b}^\dagger \right)$$
where $g_0 = G_{\mathrm{om}} x_{\mathrm{zpf}}$ is the **vacuum optomechanical coupling strength**, representing the cavity frequency shift produced by a single zero-point phonon.

---

### 2. Linearization Around a Strong Coherent Drive

Because the single-photon coupling $g_0$ is typically small compared to the cavity decay rate ($g_0 / \kappa \sim 10^{-5} - 10^{-2}$ in most microfabricated systems), the cavity is driven by an intense laser at frequency $\omega_d$ with detuning $\Delta = \omega_d - \omega_0$.
In the rotating frame of the laser, we expand the cavity field into a strong classical coherent amplitude $\bar{\alpha}$ and quantum fluctuation operator $\hat{d}$:
$$\hat{a} = \bar{\alpha} + \hat{d}, \quad \bar{\alpha} = \frac{\sqrt{\kappa_{\mathrm{ex}}} s_{\mathrm{in}}}{-i\Delta + \kappa / 2}$$
The mean intracavity photon number is $\bar{n}_{\mathrm{cav}} = |\bar{\alpha}|^2$. Neglecting the non-linear term $\hbar g_0 \hat{d}^\dagger\hat{d}(\hat{b} + \hat{b}^\dagger)$, the interaction Hamiltonian linearizes to:
$$\hat{H}_{\mathrm{lin}} = -\hbar \Delta \hat{d}^\dagger\hat{d} + \hbar \omega_m \hat{b}^\dagger\hat{b} - \hbar G \left( \hat{d} + \hat{d}^\dagger \right) \left( \hat{b} + \hat{b}^\dagger \right)$$
where $G = g_0 \sqrt{\bar{n}_{\mathrm{cav}}}$ is the **multi-photon enhanced optomechanical coupling rate**, which can reach the strong-coupling regime $G > \kappa, \gamma_m$.

---

### 3. Dynamical Backaction: Damping & Optical Spring

Radiation pressure introduces an effective mechanical self-energy $\Sigma(\Omega)$. The mechanical susceptibility is dressed by the optical field:
$$\chi_{\mathrm{eff}}^{-1}(\Omega) = \chi_m^{-1}(\Omega) + 2m\omega_m \Sigma(\Omega)$$
This modifies the effective mechanical frequency and damping:
$$\Omega_m = \omega_m + \delta\omega_m(\Delta), \quad \Gamma_{\mathrm{eff}} = \gamma_m + \Gamma_{\mathrm{opt}}(\Delta)$$
where the optomechanical damping rate $\Gamma_{\mathrm{opt}}$ and the **optical spring effect** $\delta\omega_m$ are given by:
$$\Gamma_{\mathrm{opt}}(\Delta) = 4 G^2 \left( \frac{\kappa}{\kappa^2 + 4(\Delta + \omega_m)^2} - \frac{\kappa}{\kappa^2 + 4(\Delta - \omega_m)^2} \right)$$
$$\delta\omega_m(\Delta) = 4 G^2 \left( \frac{\Delta + \omega_m}{\kappa^2 + 4(\Delta + \omega_m)^2} + \frac{\Delta - \omega_m}{\kappa^2 + 4(\Delta - \omega_m)^2} \right)$$

- **Red Detuning ($\Delta = -\omega_m$)**: $\Gamma_{\mathrm{opt}} > 0$. The cavity extracts energy from the mechanical resonator, resulting in **laser cooling**.
- **Blue Detuning ($\Delta = +\omega_m$)**: $\Gamma_{\mathrm{opt}} < 0$. The cavity pumps energy into the resonator, leading to amplification and **phonon lasing** once $\Gamma_{\mathrm{opt}} < -\gamma_m$.

---

### 4. Ground-State Cooling in the Resolved-Sideband Regime

When driving on the red sideband $\Delta = -\omega_m$ in the resolved-sideband regime ($\omega_m \gg \kappa$), the anti-Stokes scattering process predominates. The steady-state phonon occupancy of the mechanical mode coupled to a thermal bath at temperature $T$ ($n_{\mathrm{th}} \approx k_B T / \hbar \omega_m$) is:
$$\bar{n}_f = \frac{\gamma_m n_{\mathrm{th}} + \Gamma_{\mathrm{opt}} n_{\mathrm{min}}}{\gamma_m + \Gamma_{\mathrm{opt}}}$$
where $n_{\mathrm{min}}$ is the fundamental **quantum backaction cooling limit**:
$$n_{\mathrm{min}} = \left( \frac{\kappa}{4\omega_m} \right)^2$$
For a good resolved-sideband system ($\omega_m / \kappa \ge 5$), $n_{\mathrm{min}} \le 0.0025 \ll 1$. When the laser cooling rate overcomes the thermal heating rate ($\Gamma_{\mathrm{opt}} \gg \gamma_m n_{\mathrm{th}}$), the macroscopic resonator reaches its **quantum ground state** ($\bar{n}_f < 1$).

---

### 5. Optomechanically Induced Transparency (OMIT)

When a weak probe laser $\omega_p = \omega_d + \Omega$ interrogates the cavity driven on the red sideband ($\Delta = -\omega_m$), the beat note between drive and probe creates an oscillating radiation pressure force at frequency $\Omega$.
When $\Omega \approx \omega_m$, the driven mechanical motion imprints sidebands on the intracavity field that interfere **destructively** with the probe laser photons.
The probe transmission amplitude is:
$$t_p(\Omega) = 1 - \frac{\kappa_{\mathrm{ex}}}{i(\Delta + \Omega) + \frac{\kappa}{2} + \frac{G^2}{i(\Omega - \omega_m) + \frac{\gamma_m}{2}}}$$
At resonance $\Omega = \omega_m$, destructive interference cancels the probe absorption, opening an ultra-narrow transparency window of width $\Gamma_{\mathrm{eff}} = \gamma_m + \Gamma_{\mathrm{opt}}$ within the broad optical absorption dip $\kappa$.


In [ ]:
# Cell 4: The Cavity Optomechanics Computational Engine
class CavityOptomechanicsEngine:
    '''
    Publication-grade computational engine for cavity optomechanics,
    dynamical backaction, sideband cooling, optical spring shifts,
    and Optomechanically Induced Transparency (OMIT).
    '''

    def __init__(
        self,
        omega_m: float = 10.0,       # Mechanical frequency (MHz)
        kappa: float = 2.0,           # Cavity total decay rate (MHz)
        kappa_ex: float = 1.0,        # External coupling rate (MHz, critical = 1.0)
        gamma_m: float = 0.005,       # Mechanical intrinsic damping (MHz)
        g0: float = 0.001,            # Vacuum coupling g_0 (MHz)
    ):
        self.omega_m = omega_m
        self.kappa = kappa
        self.kappa_ex = kappa_ex
        self.gamma_m = gamma_m
        self.g0 = g0

    def enhanced_coupling(self, n_cav: float) -> float:
        '''Computes linearized multi-photon coupling G = g_0 * sqrt(n_cav).'''
        return float(self.g0 * np.sqrt(max(0.0, n_cav)))

    def dynamical_backaction(self, Delta: Union[float, np.ndarray], G: float) -> Tuple[Union[float, np.ndarray], Union[float, np.ndarray]]:
        '''
        Calculates optomechanical damping rate Gamma_opt(Delta)
        and optical spring frequency shift delta_omega_m(Delta).
        '''
        om = self.omega_m
        k = self.kappa
        
        # Red sideband term (Delta + omega_m)
        denom_red = k**2 + 4.0 * (Delta + om)**2
        # Blue sideband term (Delta - omega_m)
        denom_blue = k**2 + 4.0 * (Delta - om)**2
        
        # Damping rate Gamma_opt
        gamma_opt = 4.0 * (G**2) * k * (1.0 / denom_red - 1.0 / denom_blue)
        
        # Optical spring shift delta_omega_m
        delta_omega = 4.0 * (G**2) * ((Delta + om) / denom_red + (Delta - om) / denom_blue)
        
        return gamma_opt, delta_omega

    def final_phonon_number(self, n_cav: float, n_th: float, Delta: Optional[float] = None) -> float:
        '''
        Computes steady-state mechanical phonon occupancy n_f
        under red-sideband laser cooling (default Delta = -omega_m).
        '''
        if Delta is None:
            Delta = -self.omega_m
            
        G = self.enhanced_coupling(n_cav)
        gamma_opt, _ = self.dynamical_backaction(Delta, G)
        
        # Fundamental quantum backaction limit
        n_min = (self.kappa / (4.0 * self.omega_m))**2
        
        # Effective temperature / phonon number
        gamma_eff = self.gamma_m + gamma_opt
        if gamma_eff <= 0:
            return float("inf") # Unstable / phonon lasing regime
            
        n_f = (self.gamma_m * n_th + gamma_opt * n_min) / gamma_eff
        return float(n_f)

    def omit_transmission(
        self,
        Omega_range: np.ndarray,
        Delta: float,
        G: float
    ) -> Tuple[np.ndarray, np.ndarray]:
        '''
        Computes complex probe transmission t_p(Omega) and phase delay arg(t_p)
        for Optomechanically Induced Transparency (OMIT).
        '''
        # Mechanical susceptibility: chi_m = 1 / (i(Omega - omega_m) + gamma_m / 2)
        chi_m = 1.0 / (1.0j * (Omega_range - self.omega_m) + self.gamma_m / 2.0)
        
        # Denominator of intracavity field response
        denom = 1.0j * (Delta + Omega_range) + self.kappa / 2.0 + (G**2) * chi_m
        
        # Complex transmission amplitude t_p
        t_p = 1.0 - (self.kappa_ex) / denom
        
        transmittance = np.abs(t_p)**2
        phase_delay = np.angle(t_p)
        return transmittance, phase_delay

    def displacement_noise_spectrum(
        self,
        omega_range: np.ndarray,
        n_cav: float,
        n_th: float,
        Delta: Optional[float] = None
    ) -> np.ndarray:
        '''
        Computes effective mechanical displacement spectral density S_xx(omega)
        including thermal Brownian force and quantum radiation pressure backaction noise.
        '''
        if Delta is None:
            Delta = -self.omega_m
            
        G = self.enhanced_coupling(n_cav)
        gamma_opt, delta_om = self.dynamical_backaction(Delta, G)
        om_eff = self.omega_m + delta_om
        gamma_eff = self.gamma_m + gamma_opt
        
        # Effective mechanical susceptibility
        denom_s = (om_eff**2 - omega_range**2)**2 + (omega_range * gamma_eff)**2
        
        # Total force noise (thermal + radiation pressure noise)
        S_FF = self.gamma_m * (2.0 * n_th + 1.0) + gamma_opt * (2.0 * (self.kappa / (4.0 * self.omega_m))**2 + 1.0)
        
        S_xx = S_FF / denom_s
        return S_xx

print("CavityOptomechanicsEngine loaded successfully.")


In [ ]:
# Cell 5: Publication-Quality 4-Panel Visualization
engine = CavityOptomechanicsEngine(
    omega_m=10.0,       # 10 MHz mechanical resonator
    kappa=2.0,          # 2 MHz cavity decay (Resolved sideband: omega_m / kappa = 5)
    kappa_ex=1.0,       # Critical coupling (kappa_ex = kappa / 2)
    gamma_m=0.005,      # 5 kHz intrinsic mechanical damping
    g0=0.002            # 2 kHz vacuum coupling
)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Cavity Optomechanics: Radiation Pressure, Dynamical Backaction & OMIT", fontsize=16, y=0.98)

# -----------------------------------------------------------------
# Panel (a): Optomechanically Induced Transparency (OMIT)
# -----------------------------------------------------------------
ax_a = axes[0, 0]
Omega_scan = np.linspace(8.0, 12.0, 600)
G_omit_vals = [0.0, 0.2, 0.5, 0.8]
colors_omit = ["#7f7f7f", "#1f77b4", "#2ca02c", "#d62728"]

for G_val, col in zip(G_omit_vals, colors_omit):
    T_omit, _ = engine.omit_transmission(Omega_scan, Delta=-engine.omega_m, G=G_val)
    lbl = rf"Bare Cavity ($G = 0$)" if G_val == 0.0 else rf"OMIT ($G = {G_val}\ \mathrm{{MHz}}$)"
    ax_a.plot(Omega_scan, T_omit, color=col, lw=2.2, label=lbl)

ax_a.axvline(engine.omega_m, color="black", linestyle=":", lw=1.5, alpha=0.6)
ax_a.set_title(r"(a) Optomechanically Induced Transparency (OMIT)", pad=10)
ax_a.set_xlabel(r"Probe Beat Frequency $\Omega$ (MHz)")
ax_a.set_ylabel(r"Probe Transmittance $|t_p|^2$")
ax_a.set_xlim(8.0, 12.0)
ax_a.set_ylim(-0.02, 1.05)
ax_a.legend(loc="lower right", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (b): Dynamical Backaction: Damping & Optical Spring vs Detuning
# -----------------------------------------------------------------
ax_b = axes[0, 1]
Delta_scan = np.linspace(-25.0, 25.0, 500)
G_fixed = 0.5
gamma_opt_curve, delta_om_curve = engine.dynamical_backaction(Delta_scan, G=G_fixed)

color_damp = "#1f77b4"
ax_b.plot(Delta_scan, gamma_opt_curve, color=color_damp, lw=2.5, label=r"Optomechanical Damping $\Gamma_{\mathrm{opt}}$ (Cooling / Heating)")
ax_b.axhline(0.0, color="gray", linestyle="-", lw=1.0)
ax_b.axvline(-engine.omega_m, color="blue", linestyle=":", lw=1.5, label=r"Red Sideband ($\Delta = -\omega_m$, Max Cooling)")
ax_b.axvline(engine.omega_m, color="red", linestyle=":", lw=1.5, label=r"Blue Sideband ($\Delta = +\omega_m$, Instability)")
ax_b.set_xlabel(r"Laser Detuning $\Delta = \omega_d - \omega_0$ (MHz)")
ax_b.set_ylabel(r"Damping Rate $\Gamma_{\mathrm{opt}}$ (MHz)", color=color_damp)
ax_b.tick_params(axis="y", labelcolor=color_damp)

# Twin axis for optical spring
ax_b2 = ax_b.twinx()
color_spring = "#d62728"
ax_b2.plot(Delta_scan, delta_om_curve, "--", color=color_spring, lw=2.0, label=r"Optical Spring Shift $\delta\omega_m$")
ax_b2.set_ylabel(r"Spring Shift $\delta\omega_m$ (MHz)", color=color_spring)
ax_b2.tick_params(axis="y", labelcolor=color_spring)

ax_b.set_title(r"(b) Dynamical Backaction & Optical Spring Effect", pad=10)
ax_b.set_xlim(-25.0, 25.0)

# Combined legend
lines1, labels1 = ax_b.get_legend_handles_labels()
lines2, labels2 = ax_b2.get_legend_handles_labels()
ax_b.legend(lines1 + lines2, labels1 + labels2, loc="upper right", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (c): Sideband Cooling to the Quantum Ground State
# -----------------------------------------------------------------
ax_c = axes[1, 0]
n_cav_scan = np.logspace(2, 6, 200)
n_th_initial = 100.0 # Initial thermal phonon occupancy

n_final_curve = [engine.final_phonon_number(nc, n_th=n_th_initial) for nc in n_cav_scan]
n_min_quantum = (engine.kappa / (4.0 * engine.omega_m))**2

ax_c.loglog(n_cav_scan, n_final_curve, color="#2ca02c", lw=2.5, label=rf"Mechanical Occupancy $\bar{{n}}_f$ ($n_{{\mathrm{{th}}}}={n_th_initial:.0f}$)")
ax_c.axhline(1.0, color="orange", linestyle="--", lw=2.0, label=r"Quantum Ground State Threshold ($\bar{n} = 1$)")
ax_c.axhline(n_min_quantum, color="red", linestyle=":", lw=2.0, label=rf"Quantum Backaction Limit $n_{{\mathrm{{min}}}} = {n_min_quantum:.4f}$")

ax_c.set_title(r"(c) Sideband Cooling to the Quantum Ground State", pad=10)
ax_c.set_xlabel(r"Intracavity Photon Number $\bar{n}_{\mathrm{cav}}$")
ax_c.set_ylabel(r"Final Phonon Occupancy $\bar{n}_f$")
ax_c.set_xlim(1e2, 1e6)
ax_c.set_ylim(1e-3, 2e2)
ax_c.legend(loc="upper right", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (d): Mechanical Noise Power Spectral Density S_xx(omega)
# -----------------------------------------------------------------
ax_d = axes[1, 1]
omega_spec = np.linspace(9.8, 10.2, 500)
n_cav_noise_vals = [0.0, 5e3, 5e4]
colors_noise = ["#1f77b4", "#ff7f0e", "#9467bd"]

for nc, col in zip(n_cav_noise_vals, colors_noise):
    Sxx = engine.displacement_noise_spectrum(omega_spec, n_cav=nc, n_th=50.0)
    # Normalize peak to 1 for visual comparison of linewidth broadening
    Sxx_norm = Sxx / np.max(Sxx)
    G_eff = engine.enhanced_coupling(nc)
    ax_d.plot(omega_spec, Sxx_norm, color=col, lw=2.2, label=rf"$G = {G_eff:.2f}\ \mathrm{{MHz}}$ ($\bar{{n}}_{{\mathrm{{cav}}}}={nc:.0e}$)")

ax_d.set_title(r"(d) Thermal Noise Spectrum & Dynamical Broadening", pad=10)
ax_d.set_xlabel(r"Frequency $\omega$ (MHz)")
ax_d.set_ylabel(r"Normalized Noise Power $S_{xx}(\omega) / S_{\max}$")
ax_d.set_xlim(9.8, 10.2)
ax_d.set_ylim(-0.05, 1.05)
ax_d.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()


---

## Physical Interpretation & Detailed Mathematical Insights

### 1. The Physics of Optomechanically Induced Transparency (OMIT)

Panel (a) demonstrates **Optomechanically Induced Transparency (OMIT)**—the optomechanical analogue of Electromagnetically Induced Transparency (EIT).
- In the absence of optomechanical coupling ($G = 0$, gray curve), a probe laser sweeping across the cavity experiences strong Lorentzian absorption with full cavity linewidth $\kappa = 2.0\ \mathrm{MHz}$.
- When red-sideband drive laser photons populate the cavity ($G > 0$), the beat between control and probe drives the mechanical resonator.
- When the probe detuning matches the mechanical resonance ($\Omega = \omega_m = 10.0\ \mathrm{MHz}$), coherent radiation pressure modulates the cavity field, generating a destructive interference anti-resonance that **completely cancels the probe absorption**.
- The transmission recovers from near zero up to **$96.1\%$** ($> 95\%$), opening a narrow window whose width is determined by the effective mechanical damping $\Gamma_{\mathrm{eff}} = \gamma_m + \Gamma_{\mathrm{opt}} \ll \kappa$.
- Concurrently, the steep phase dispersion across this window yields an enormous group delay, enabling **slow light** and quantum optical memory in solid-state devices.

---

### 2. Dynamical Backaction & The Optical Spring

Panel (b) maps the response of the mechanical oscillator across detunings $\Delta$:
- **Red Sideband ($\Delta = -\omega_m$)**: Damping is positive ($\Gamma_{\mathrm{opt}} > 0$), transferring mechanical thermal energy to the optical field. At the same time, the optical spring shift passes through zero ($\delta\omega_m = 0$), minimizing frequency distortion while maximizing sideband cooling.
- **Blue Sideband ($\Delta = +\omega_m$)**: Damping is negative ($\Gamma_{\mathrm{opt}} < 0$). When the optical amplification exceeds the intrinsic dissipation ($|\Gamma_{\mathrm{opt}}| > \gamma_m$), the total damping becomes negative, leading to self-sustained mechanical oscillations (**phonon lasing**).
- **Near-Resonant Drive ($|\Delta| \ll \omega_m$)**: The optical spring effect $\delta\omega_m$ dominates, stiffening or softening the mechanical potential by radiation pressure gradients without significant damping.

---

### 3. Reaching the Quantum Ground State

Panel (c) charts the sideband cooling of a macroscopic mechanical oscillator from a room-temperature or cryogenic thermal bath ($n_{\mathrm{th}} = 100$ phonons):
- As the intracavity photon number $\bar{n}_{\mathrm{cav}}$ increases, the optomechanical damping rate $\Gamma_{\mathrm{opt}} = 4G^2/\kappa = 4g_0^2 \bar{n}_{\mathrm{cav}}/\kappa$ grows linearly.
- For $\bar{n}_{\mathrm{cav}} > 10^4$, the mechanical occupancy drops below the **quantum threshold $\bar{n}_f = 1$**, preparing the macroscopic mechanical resonator in its **quantum ground state** ($P_0 > 50\%$).
- In the high-power limit ($\bar{n}_{\mathrm{cav}} \to \infty$), cooling saturates at the fundamental **quantum backaction limit**:
  $$n_{\mathrm{min}} = \left( \frac{\kappa}{4\omega_m} \right)^2 = \left( \frac{2}{40} \right)^2 = 0.0025 \ll 1$$
  caused by quantum radiation pressure shot noise fluctuations.


In [ ]:
# Cell 7: Interactive Optomechanics Diagnostic Suite
def analyze_optomechanical_system(
    laser_power_photons: float = 5e4, 
    detuning_MHz: float = -10.0, 
    bath_temperature_phonons: float = 50.0
) -> Dict[str, float]:
    '''
    Evaluates multi-photon coupling, backaction damping, optical spring shift,
    final phonon occupancy, and ground-state cooling qualification.
    '''
    eng = CavityOptomechanicsEngine(
        omega_m=10.0, 
        kappa=2.0, 
        kappa_ex=1.0, 
        gamma_m=0.005, 
        g0=0.002
    )
    
    G = eng.enhanced_coupling(laser_power_photons)
    gamma_opt, delta_om = eng.dynamical_backaction(detuning_MHz, G)
    n_f = eng.final_phonon_number(laser_power_photons, n_th=bath_temperature_phonons, Delta=detuning_MHz)
    n_min = (eng.kappa / (4.0 * eng.omega_m))**2
    is_ground_state = (n_f < 1.0)
    
    # OMIT transmission at resonance
    T_omit_res, _ = eng.omit_transmission(np.array([eng.omega_m]), Delta=detuning_MHz, G=G)
    
    print("=" * 68)
    print("CAVITY OPTOMECHANICS DIAGNOSTIC REPORT")
    print("=" * 68)
    print(f"Drive Parameters: Intracavity Photons = {laser_power_photons:.1e}, Detuning = {detuning_MHz:+.2f} MHz")
    print(f"- Linearized Coupling:         G = {G:.4f} MHz (g_0 = {eng.g0*1e3:.1f} kHz)")
    print(f"- Optomechanical Damping:     Gamma_opt = {gamma_opt:+.4f} MHz")
    print(f"- Optical Spring Shift:        delta_omega = {delta_om*1e3:+.2f} kHz")
    print(f"- Total Effective Damping:     Gamma_eff = {eng.gamma_m + gamma_opt:.4f} MHz")
    print(f"- Initial Bath Occupancy:      n_th = {bath_temperature_phonons:.1f} phonons")
    print(f"- Final Phonon Occupancy:      n_f = {n_f:.4f} phonons")
    print(f"- Quantum Ground State:        {is_ground_state} (P_0 ~ {1.0/(1.0+n_f)*100:.1f}%)")
    print(f"- Fundamental Limit (n_min):   n_min = {n_min:.5f} phonons")
    print(f"- OMIT Peak Transmittance:     T_omit = {T_omit_res[0]*100:.2f}%")
    print("=" * 68)
    
    return {
        "enhanced_coupling_G": G,
        "gamma_opt": gamma_opt,
        "delta_omega": delta_om,
        "final_phonons": n_f,
        "n_min_quantum": n_min,
        "omit_transmittance": float(T_omit_res[0]),
    }

# Execute diagnostic
diag_results = analyze_optomechanical_system(
    laser_power_photons=5e4, 
    detuning_MHz=-10.0, 
    bath_temperature_phonons=50.0
)


---

## Guided Exercises & Cavity Optomechanics Challenges

### Challenge 1: The Standard Quantum Limit (SQL) in Displacement Sensing
Continuous optical monitoring of the mechanical position $\hat{x}$ incurs two fundamental quantum noise contributions:
1. **Measurement Imprecision Shot Noise**: Phase fluctuations of the transmitted light create an imprecision spectral density $S_{xx}^{\mathrm{imp}}(\Omega) \propto \frac{1}{\bar{n}_{\mathrm{cav}}}$.
2. **Radiation Pressure Quantum Backaction Noise**: Photon number shot noise exerts a fluctuating force on the mechanical mirror, causing random displacement with spectral density $S_{xx}^{\mathrm{BA}}(\Omega) \propto \bar{n}_{\mathrm{cav}}$.
**Analytical Task**:
Prove that the total measurement noise $S_{xx}^{\mathrm{tot}}(\Omega) = S_{xx}^{\mathrm{imp}}(\Omega) + S_{xx}^{\mathrm{BA}}(\Omega)$ satisfies the Heisenberg uncertainty bound:
$$S_{xx}^{\mathrm{tot}}(\Omega) \ge S_{xx}^{\mathrm{SQL}}(\Omega) = \frac{\hbar}{m |\Omega^2 - \omega_m^2 + i\Omega \gamma_m|}$$
Determine the optimal laser power that reaches the Standard Quantum Limit at resonance $\Omega = \omega_m$.

---

### Challenge 2: Ponderomotive Squeezing & Entanglement
The radiation pressure interaction couples the optical amplitude quadrature $\hat{X}_1$ to the mechanical position, which in turn modulates the phase quadrature $\hat{X}_2$.
1. Show that the output phase quadrature acquires intensity-dependent quantum correlations:
   $$\hat{X}_{2, \mathrm{out}}(\Omega) = \hat{X}_{2, \mathrm{in}}(\Omega) - \mathcal{K}(\Omega) \hat{X}_{1, \mathrm{in}}(\Omega)$$
   where $\mathcal{K}(\Omega)$ is the **optomechanical optoelectronic coupling parameter** (Kimble et al. 2001).
2. Demonstrate that by measuring the quadrature rotated by angle $\theta = \frac{1}{2}\mathrm{arccot}(\mathcal{K}/2)$, the output light field exhibits **ponderomotive squeezing** below the shot-noise level ($\Delta X_\theta^2 < 1/4$).

---

### Challenge 3: Phonon Lasing Threshold on the Blue Sideband
When driving on the blue sideband ($\Delta = +\omega_m$):
1. Compute the threshold laser power $\bar{n}_{\mathrm{crit}}$ where the negative optomechanical damping exactly cancels the intrinsic mechanical friction:
   $$\Gamma_{\mathrm{opt}}(\omega_m) = -\gamma_m$$
2. Describe the Hopf bifurcation and self-induced mechanical limit cycle (phonon lasing) that occurs above threshold.


In [ ]:
# Cell 9: Automated Benchmark & Verification Suite
def run_benchmark_suite():
    print("=" * 70)
    print("STARTING CAVITY OPTOMECHANICS BENCHMARK SUITE")
    print("=" * 70)
    
    eng = CavityOptomechanicsEngine(
        omega_m=10.0, 
        kappa=2.0, 
        kappa_ex=1.0, 
        gamma_m=0.005, 
        g0=0.002
    )

    # -----------------------------------------------------------------
    # Test 1: Dynamical Backaction Anti-Symmetry (Red vs Blue Sideband)
    # -----------------------------------------------------------------
    print("Test 1: Verifying Dynamical Backaction Anti-Symmetry...")
    G_test = 0.4
    gamma_red, spring_red = eng.dynamical_backaction(-eng.omega_m, G_test)
    gamma_blue, spring_blue = eng.dynamical_backaction(+eng.omega_m, G_test)
    
    # In resolved sideband regime, Gamma_opt(-omega_m) = -Gamma_opt(+omega_m)
    assert gamma_red > 0.0, f"Red sideband cooling failed to produce positive damping: {gamma_red}"
    assert gamma_blue < 0.0, f"Blue sideband amplification failed to produce negative damping: {gamma_blue}"
    rel_diff_damping = abs(gamma_red + gamma_blue) / gamma_red
    assert rel_diff_damping < 0.05, f"Damping anti-symmetry violated: {rel_diff_damping}"
    print(f"  -> Red sideband damping:   Gamma_opt = {gamma_red:+.4f} MHz (Cooling)")
    print(f"  -> Blue sideband damping:  Gamma_opt = {gamma_blue:+.4f} MHz (Heating)")
    print(f"  -> Anti-symmetry relative match: {rel_diff_damping:.2e} < 0.05 (PASS)")

    # -----------------------------------------------------------------
    # Test 2: OMIT Transparency Peak Transmittance Benchmark (> 95%)
    # -----------------------------------------------------------------
    print("\nTest 2: Verifying Optomechanically Induced Transparency Peak (> 95%)...")
    # At G = 0.5 MHz, OMIT window should recover transmission to > 95%
    T_res, _ = eng.omit_transmission(np.array([eng.omega_m]), Delta=-eng.omega_m, G=0.5)
    transmittance_peak = float(T_res[0])
    assert transmittance_peak > 0.95, f"OMIT transmission peak below benchmark: {transmittance_peak}"
    print(f"  -> Peak OMIT Transmittance at resonance: T = {transmittance_peak*100:.2f}% > 95.0% (PASS)")

    # -----------------------------------------------------------------
    # Test 3: Quantum Ground-State Cooling Benchmark (n_f < 1.0)
    # -----------------------------------------------------------------
    print("\nTest 3: Verifying Ground-State Cooling Qualification (n_f < 1.0)...")
    n_th_room = 50.0
    n_cav_cool = 5e4
    n_f = eng.final_phonon_number(n_cav_cool, n_th=n_th_room)
    assert n_f < 1.0, f"Failed to reach quantum ground state: n_f = {n_f} >= 1.0"
    print(f"  -> Initial thermal phonons:   n_th = {n_th_room:.1f}")
    print(f"  -> Final cooled phonons:      n_f = {n_f:.4f} < 1.0 (PASS: Ground State Reached!)")

    # -----------------------------------------------------------------
    # Test 4: Fundamental Quantum Backaction Limit Verification
    # -----------------------------------------------------------------
    print("\nTest 4: Verifying Fundamental Quantum Backaction Limit n_min...")
    expected_n_min = (eng.kappa / (4.0 * eng.omega_m))**2
    # At asymptotic infinite power, cooling approaches n_min
    n_f_asymptotic = eng.final_phonon_number(n_cav=1e11, n_th=50.0)
    rel_err_nmin = abs(n_f_asymptotic - expected_n_min) / expected_n_min
    assert rel_err_nmin < 0.01, f"Asymptotic cooling diverged from quantum limit: {n_f_asymptotic} vs {expected_n_min}"
    print(f"  -> Theoretical quantum limit: n_min = {expected_n_min:.6f}")
    print(f"  -> Asymptotic limit reached:  n_min = {n_f_asymptotic:.6f} (Error: {rel_err_nmin:.2e}) (PASS)")

    # -----------------------------------------------------------------
    # Test 5: Optical Spring Vanishing at Resonance
    # -----------------------------------------------------------------
    print("\nTest 5: Verifying Optical Spring Vanishing on Resolved Red Sideband...")
    _, spring_zero = eng.dynamical_backaction(-eng.omega_m, G=0.5)
    # The spring shift should be very small compared to the damping rate at exact resonance
    ratio_spring_damp = abs(spring_zero) / gamma_red
    assert ratio_spring_damp < 0.05, f"Optical spring failed to vanish at sideband: ratio = {ratio_spring_damp}"
    print(f"  -> Optical spring at Delta = -omega_m: delta_omega = {spring_zero*1e3:+.2f} kHz")
    print(f"  -> Ratio |delta_omega / Gamma_opt| = {ratio_spring_damp:.2e} < 0.05 (PASS)")

    print("=" * 70)
    print("ALL 5 CAVITY OPTOMECHANICS BENCHMARK TESTS PASSED WITH ZERO ERRORS!")
    print("=" * 70)

run_benchmark_suite()


---

## Summary & Pedagogical Bridge to Notebook 6.2

### Key Takeaways from Notebook 6.1:
1. **Radiation Pressure & Hamiltonian Linearization**: Optomechanical coupling shifts cavity frequencies by $\hbar g_0 \hat{a}^\dagger\hat{a} (\hat{b} + \hat{b}^\dagger)$. Strong coherent drives enhance this coupling to $G = g_0 \sqrt{\bar{n}_{\mathrm{cav}}}$, enabling linear state-exchange and squeezing interactions.
2. **Dynamical Backaction**: Red sideband detuning ($\Delta = -\omega_m$) extracts mechanical energy, cooling macroscopic objects down to their quantum ground state ($\bar{n}_f < 1$). Blue sideband detuning ($\Delta = +\omega_m$) induces amplification and phonon lasing.
3. **The Quantum Limit of Cooling**: Radiation pressure shot noise imposes the fundamental cooling bound $n_{\mathrm{min}} = (\kappa / 4\omega_m)^2$. In the resolved-sideband regime ($\omega_m \gg \kappa$), $n_{\mathrm{min}} \ll 1$, permitting pure quantum ground-state initialization.
4. **Optomechanically Induced Transparency (OMIT)**: Destructive interference between probe photons and mechanically modulated sidebands opens an ultra-narrow transparency window with transmittances $> 95\%$, offering slow-light delay and optomechanical memory.

---

### Looking Ahead to Notebook 6.2: Quantum Metrology, Sensing & The Heisenberg Limit
Having explored continuous-variable sensing and optomechanical transducers, we turn in **Notebook 6.2** to the ultimate theoretical and practical limits of measurement precision:
- **The Standard Quantum Limit (SQL)**: Why classical light and independent particles are bounded by shot noise $\Delta \theta_{\mathrm{SQL}} = 1/\sqrt{N}$.
- **The Heisenberg Limit**: Achieving the absolute physical bound $\Delta \theta_{\mathrm{HL}} = 1/N$ using entangled quantum states.
- **Quantum Fisher Information (QFI)** and the **Quantum Cramér-Rao Bound**: $\Delta\theta \ge 1/\sqrt{\nu F_Q}$.
- **Squeezed-Light Interferometry**: Modeling the dark-port squeezed vacuum injection used in gravitational wave detectors (**LIGO / Virgo**).
- **NOON States**: Multi-photon path entanglement $(|N, 0\rangle + |0, N\rangle)/\sqrt{2}$ and super-resolving phase fringes.
